# 视频 → 人体姿态 JSON / NPZ → Blender

从上到下执行：配置 → 克隆本项目 → pip/uv 安装依赖 → 克隆并补丁 SAM/DINOv3 → 下载模型 → 检查与提取。
准备步骤直接执行命令，不需要另外调用安装/克隆/下载封装函数。命令通过 `subprocess.run(..., check=True)` 执行，失败会停止当前单元，路径含空格也可使用。

使用 Python 3.11（也支持 3.12）kernel，先按 README 准备匹配硬件的 PyTorch/torchvision。默认 GPU、多人；最后一格默认逐帧处理完整视频，试跑可设置 `MAX_FRAMES = 30`；CPU 请将 `DEVICE` 改为 `cpu`。
每个准备单元执行时才会联网或安装；复用已有资源时会校验，不执行 pull/reset。安装后重启 kernel，再运行配置和项目定位单元，跳过依赖安装单元继续。
模型加载后，只需修改并运行最后一格中的 `VIDEOS`（单个路径或视频列表），即可继续处理；每次自动创建新的输出目录。

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# 在项目根目录/extractor 目录运行时直接复用；只上传此 notebook 时克隆到当前目录。
PROJECT_URL = "https://github.com/fjlikun/pose_pipeline.git"
PROJECT_BRANCH = "develop"  # 默认仅克隆不含 MP4 的代码分支
WORK_DIR = Path.cwd().resolve()
if (WORK_DIR / "extractor" / "extract_video.py").is_file():
    PROJECT_DIR = WORK_DIR
elif (WORK_DIR / "extract_video.py").is_file() and WORK_DIR.name == "extractor":
    PROJECT_DIR = WORK_DIR.parent
else:
    PROJECT_DIR = WORK_DIR / "pose_pipeline"
EXTRACTOR_DIR = PROJECT_DIR / "extractor"
RUNTIME_DIR = PROJECT_DIR / "runtime"
PACKAGE_MANAGER = "pip"  # 可改为 "uv"；uv 命令需要预先安装
DEVICE = "cuda"          # NVIDIA CUDA / AMD ROCm 均用 cuda；CPU 用 cpu
MULTI_PERSON = True

SAM_REPO = RUNTIME_DIR / "sam-3d-body"
DINOV3_REPO = RUNTIME_DIR / "dinov3"
CHECKPOINT_DIR = RUNTIME_DIR / "checkpoints"
CHECKPOINT = CHECKPOINT_DIR / "model.ckpt"
MHR_MODEL = CHECKPOINT_DIR / "assets" / "mhr_model.pt"
DETECTOR_WEIGHTS = RUNTIME_DIR / "detector" / "fasterrcnn_resnet50_fpn_v2_coco.pth"
MODEL_SOURCE = "modelscope"  # 或 "huggingface"；官方源需提前获得访问权限
MODEL_REVISION = "master" if MODEL_SOURCE == "modelscope" else "main"

print("Python:", sys.executable)
print("项目:", PROJECT_DIR)

## 1. git clone 本项目

已有完整项目时直接复用；空环境中仅浅克隆 `PROJECT_BRANCH`（默认 develop）：`git clone --branch develop --single-branch --depth 1`。如果仓库为私有，需要预先配置 Git 访问权限，或修改 `PROJECT_URL`。develop 不包含 MP4 或 Git LFS 资源规则，推理使用最后一格指定的自己的视频。

In [ ]:
if not (EXTRACTOR_DIR / "extract_video.py").is_file():
    if PROJECT_DIR.exists():
        raise FileExistsError(f"已有目录不是完整项目，请修改 PROJECT_DIR: {PROJECT_DIR}")
    PROJECT_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--branch", PROJECT_BRANCH, "--single-branch",
                    "--depth", "1", PROJECT_URL, str(PROJECT_DIR)], check=True)
if not (EXTRACTOR_DIR / "requirements.txt").is_file():
    raise FileNotFoundError(EXTRACTOR_DIR / "requirements.txt")
if str(EXTRACTOR_DIR) not in sys.path:
    sys.path.insert(0, str(EXTRACTOR_DIR))
os.environ["SAM3D_DINOV3_REPO"] = str(DINOV3_REPO)
print("使用项目:", PROJECT_DIR)

## 2. 通过 requirements.txt 安装依赖（pip / uv）

`extractor/requirements.txt` 包含固定 SAM 版本的 Python 依赖、PyAV、NumPy、Notebook 及模型下载客户端。它不安装 PyTorch/torchvision/CUDA，请先选好已有运行环境。

实际命令为 `当前Python -m pip install -r requirements.txt -c torch-constraints.txt`，或 `uv pip install --python 当前Python -r requirements.txt -c torch-constraints.txt`。
下面固定现有 torch/CUDA 相关包版本，其他依赖仍按 pip/uv 正常解析，可能调整版本；依赖冲突会直接报错。

默认不安装推理未使用的 `xtcocotools`，避免其源码构建时因缺少 NumPy 构建依赖声明而失败。安装后重启 kernel，再执行配置和项目定位单元，然后跳过本安装单元继续。其他包从源码构建时可能需要 C/C++ 编译工具，详见 README。

In [ ]:
from importlib.metadata import distributions
import re
import shutil

if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError("请选择 Python 3.11/3.12 kernel")
versions = {}
for dist in distributions():
    if dist.metadata.get("Name"):
        name = re.sub(r"[-_.]+", "-", dist.metadata["Name"]).lower()
        versions[name] = dist.version
if not all(name in versions for name in ("torch", "torchvision")):
    raise RuntimeError("请先按 README 安装匹配 CPU/CUDA 的 torch 和 torchvision，再执行本单元")
if PACKAGE_MANAGER == "pip":
    install_cmd = [sys.executable, "-m", "pip", "install"]
    check_cmd = [sys.executable, "-m", "pip", "check"]
elif PACKAGE_MANAGER == "uv":
    uv = shutil.which("uv")
    if uv is None:
        raise RuntimeError("未找到 uv；请先安装 uv，或设置 PACKAGE_MANAGER='pip'")
    install_cmd = [uv, "pip", "install", "--python", sys.executable]
    check_cmd = [uv, "pip", "check", "--python", sys.executable]
else:
    raise ValueError("PACKAGE_MANAGER 必须为 pip 或 uv")
protected = {name: version for name, version in versions.items()
             if name in {"torch", "torchvision", "torchaudio", "torchtext", "torchdata", "triton", "cuda", "cudnn", "nvidia", "pytorch"}
             or name.startswith(("nvidia-", "cuda-", "cudnn", "pytorch-triton", "torch-"))}
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
constraints = RUNTIME_DIR / "torch-constraints.txt"
constraints.write_text("".join(f"{name}=={version}\n" for name, version in sorted(protected.items())), encoding="utf-8")
command = [*install_cmd, "-r", str(EXTRACTOR_DIR / "requirements.txt"), "-c", str(constraints)]
print(subprocess.list2cmdline(command))
subprocess.run(command, check=True)
subprocess.run(check_cmd, check=True)
print("依赖安装完成。请重启 kernel，重跑配置和项目定位单元，再跳过本单元继续。")

## 3. git clone 固定版本的 SAM / DINOv3

首次从 Gitee 镜像执行 `git clone --no-checkout` 和 `git checkout --detach <commit>`：SAM 使用 `https://gitee.com/plab/sam-3d-body.git`，DINOv3 使用 `https://gitee.com/nandy2022/dinov3.git`。已有目录核对配置的 origin、固定 commit 和修改，不 pull、不 reset；旧 GitHub checkout 可按 README 的 `git remote set-url` 步骤迁移后复用。

In [ ]:
from runtime_checks import SAM_URL, SAM_COMMIT, DINO_URL, DINO_COMMIT

for repo, url, commit in ((SAM_REPO, SAM_URL, SAM_COMMIT), (DINOV3_REPO, DINO_URL, DINO_COMMIT)):
    if repo.exists():
        if not (repo / ".git").exists():
            raise RuntimeError(f"不是 git checkout，请选择新路径: {repo}")
    else:
        repo.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--no-checkout", url, str(repo)], check=True)
        subprocess.run(["git", "-C", str(repo), "checkout", "--detach", commit], check=True)
    actual = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
    origin = subprocess.check_output(["git", "-C", str(repo), "remote", "get-url", "origin"], text=True).strip()
    if actual != commit or origin.rstrip("/") != url:
        raise RuntimeError(f"已有源码版本或 origin 不匹配: {repo}；期望 commit={commit}, origin={url}。旧 GitHub checkout 请按 README 的 remote set-url 步骤迁移，版本不符时使用新目录。")
if subprocess.check_output(["git", "-C", str(DINOV3_REPO), "status", "--porcelain", "--untracked-files=no"], text=True).strip():
    raise RuntimeError("DINOv3 有 tracked 修改，请检查或使用新目录")
print("SAM 和 DINOv3 固定版本已就绪")

### 应用本地 DINOv3 / checkpoint 补丁；CPU 自动追加设备补丁

先执行 `git apply --check`，成功再 `git apply`。已应用相同补丁时跳过；未知修改报错。`DEVICE='cpu'` 时再应用 `cpu_device.patch`，仅修改模型设备选择，不安装 CPU torch。改变源码设备补丁后请重启 kernel。

In [ ]:
from runtime_checks import normalized_patch

if DEVICE not in ("cpu", "cuda"):
    raise ValueError("DEVICE 必须为 cpu 或 cuda")
gpu_patch = EXTRACTOR_DIR / "gpu_local_dinov3.patch"
full_patch = EXTRACTOR_DIR / "cpu_local_dinov3.patch"
current = normalized_patch(subprocess.check_output(["git", "-C", str(SAM_REPO), "diff", "HEAD"], text=True))
if current not in ("", normalized_patch(gpu_patch.read_text(encoding="utf-8")), normalized_patch(full_patch.read_text(encoding="utf-8"))):
    raise RuntimeError("SAM 有未知修改，停止应用补丁")
if not current:
    subprocess.run(["git", "-C", str(SAM_REPO), "apply", "--check", str(gpu_patch)], check=True)
    subprocess.run(["git", "-C", str(SAM_REPO), "apply", str(gpu_patch)], check=True)
if DEVICE == "cpu":
    current = normalized_patch(subprocess.check_output(["git", "-C", str(SAM_REPO), "diff", "HEAD"], text=True))
    if current != normalized_patch(full_patch.read_text(encoding="utf-8")):
        cpu_patch = EXTRACTOR_DIR / "cpu_device.patch"
        subprocess.run(["git", "-C", str(SAM_REPO), "apply", "--check", str(cpu_patch)], check=True)
        subprocess.run(["git", "-C", str(SAM_REPO), "apply", str(cpu_patch)], check=True)
os.environ["SAM3D_DINOV3_REPO"] = str(DINOV3_REPO)
print("补丁已就绪:", DEVICE)

## 4. 命令行下载 SAM 模型（ModelScope / Hugging Face）

默认执行 ModelScope 1.29.2 的 `python -m modelscope.cli.cli download`；可切换为 `hf download`（通过当前 Python 调用 hf CLI 入口，避免 PATH 指向其他环境）。只下载 `model.ckpt`、`model_config.yaml`、`assets/mhr_model.pt`。

请先阅读 [模型及源许可](https://huggingface.co/facebook/sam-3d-body-dinov3)。使用官方 Hugging Face 时需已获授权，并在终端 `hf auth login` 或 Notebook 平台的 secret 环境变量中配置凭证，不把 token 写入单元格。
下载约 2GB；依赖、缓存和模型建议预留 15–20GB 以上空间。先在临时目录下载、校验，再发布到 CHECKPOINT_DIR；已有目录只校验，不覆盖。

In [ ]:
import json
import shutil
import tempfile
from runtime_checks import MODEL_ID, MODEL_FILES, KNOWN_HASHES, verify_assets

if MODEL_SOURCE not in ("modelscope", "huggingface"):
    raise ValueError("MODEL_SOURCE 必须为 modelscope 或 huggingface")
if CHECKPOINT_DIR.exists():
    sam_assets = verify_assets(CHECKPOINT_DIR, KNOWN_HASHES)
    print("复用已校验模型:", CHECKPOINT_DIR)
else:
    CHECKPOINT_DIR.parent.mkdir(parents=True, exist_ok=True)
    if shutil.disk_usage(CHECKPOINT_DIR.parent).free < 8 * 1024**3:
        raise RuntimeError("SAM 下载至少需要 8 GiB 可用空间")
    with tempfile.TemporaryDirectory(prefix="sam-cli-", dir=CHECKPOINT_DIR.parent) as temporary:
        staging = Path(temporary) / "checkpoints"
        if MODEL_SOURCE == "modelscope":
            command = [sys.executable, "-m", "modelscope.cli.cli", "download", "--model", MODEL_ID,
                       "--revision", MODEL_REVISION, "--local_dir", str(staging), "--include", *MODEL_FILES]
        else:
            # 等价于 hf download；使用当前 kernel 中安装的 huggingface_hub CLI。
            command = [sys.executable, "-c", "from huggingface_hub.cli.hf import main; main()",
                       "download", MODEL_ID, *MODEL_FILES, "--revision", MODEL_REVISION, "--local-dir", str(staging)]
        print(subprocess.list2cmdline(command))
        download_env = dict(os.environ)
        if MODEL_SOURCE == "modelscope":
            download_env["MODELSCOPE_DOWNLOAD_PARALLELS"] = "8"
        subprocess.run(command, env=download_env, check=True)
        sam_assets = verify_assets(staging, KNOWN_HASHES)
        (staging / "download_provenance.json").write_text(json.dumps(dict(
            source=MODEL_SOURCE, model_id=MODEL_ID, requested_revision=MODEL_REVISION,
            files=sam_assets), indent=2), encoding="utf-8")
        if CHECKPOINT_DIR.exists():
            raise FileExistsError("模型目标目录已出现，停止发布")
        staging.rename(CHECKPOINT_DIR)
print("SAM 模型已就绪:", CHECKPOINT_DIR)

## 5. 下载多人检测器权重（自动多人使用；手动框可跳过）

从当前 torchvision 的官方权重枚举读取 URL，使用 Python 命令调用 `torch.hub.download_url_to_file` 并检查 URL 中的 SHA256 前缀。已有文件只校验，不覆盖。
只使用多人手动框时可跳过本单元；最后一格设置 `MULTI_BBOXES` 或逐视频 `bboxes` 即可。单人模式自动跳过。
[torchvision 代码许可](https://github.com/pytorch/vision/blob/main/LICENSE)与[预训练权重条款](https://github.com/pytorch/vision#pre-trained-model-license)请分别阅读。

In [ ]:
if MULTI_PERSON:
    import re
    import shutil
    import tempfile
    from urllib.parse import urlparse
    from torchvision.models.detection import FasterRCNN_ResNet50_FPN_V2_Weights
    from runtime_checks import sha256

    url = FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1.url
    parsed = urlparse(url)
    match = re.search(r"-([a-f0-9]{8,64})\.pth$", parsed.path)
    if parsed.scheme != "https" or parsed.hostname != "download.pytorch.org" or not match:
        raise RuntimeError("检测器权重来源或 hash 前缀不符合预期")
    prefix = match.group(1)
    if DETECTOR_WEIGHTS.exists():
        if not DETECTOR_WEIGHTS.is_file() or not sha256(DETECTOR_WEIGHTS).startswith(prefix):
            raise RuntimeError("已有检测器权重校验失败，请检查文件或选择新路径")
    else:
        DETECTOR_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
        if shutil.disk_usage(DETECTOR_WEIGHTS.parent).free < 1024**3:
            raise RuntimeError("检测器下载至少需要 1 GiB 可用空间")
        with tempfile.TemporaryDirectory(prefix="detector-cli-", dir=DETECTOR_WEIGHTS.parent) as temporary:
            candidate = Path(temporary) / "weights.pth"
            command = [sys.executable, "-c",
                       "import sys, torch; torch.hub.download_url_to_file(sys.argv[1], sys.argv[2], hash_prefix=sys.argv[3])",
                       url, str(candidate), prefix]
            print(subprocess.list2cmdline(command))
            subprocess.run(command, check=True)
            if not sha256(candidate).startswith(prefix):
                raise RuntimeError("检测器权重 SHA256 校验失败")
            with candidate.open("rb") as source, DETECTOR_WEIGHTS.open("xb") as target:
                shutil.copyfileobj(source, target)
    print("检测器权重已就绪:", DETECTOR_WEIGHTS)
else:
    print("单人模式，不需要检测器权重")

## 6. 本地校验与模型加载（同一 kernel 只需执行一次）

检查源码 commit/补丁、模型文件及 GPU，然后只加载 SAM；本单元不依赖视频路径、输出目录或检测器权重。AMD ROCm 也使用 `DEVICE = 'cuda'`。
自动多人模式在最后一格首次需要时加载检测器，之后复用；GPU 自动检测仍要求对应 CUDA/ROCm 的 GPU NMS，缺少算子时停止，不切 CPU。手动框模式完全不使用检测器和其 NMS。
换视频、改人数上限、检测阈值、手动框或采样参数，只需重跑最后一格。重启 kernel 或改变设备/模型路径时，再执行相关准备和模型加载单元。

In [ ]:
from runtime_checks import check_runtime
from extract_video import load_estimator
import torch

validated_assets = check_runtime(SAM_REPO, DINOV3_REPO, CHECKPOINT_DIR, target=DEVICE, expected_hashes=KNOWN_HASHES)
# 手动框模式不需要检测器；自动模式首次运行最后一格时再加载并复用。
detector = None
estimator, runtime_metadata = load_estimator(SAM_REPO, CHECKPOINT, MHR_MODEL, DEVICE)
runtime_metadata["validated_assets"] = validated_assets
print(runtime_metadata)
print("SAM 已加载。最后一格可设置每个视频的人数上限或多人固定框；自动检测器按需加载。")

## 7. 处理视频（最后一格，可重复执行）

只需修改下面一格并执行：`VIDEOS` 支持字符串/Path、路径列表，或逐视频字典 `{'video': 路径, 'max_people': 2, 'bboxes': None}`。省略可选字段时继承本格默认值，按顺序处理并复用模型。
`MAX_PEOPLE` 为自动检测每帧最多保留人数，`DETECTOR_THRESHOLD` 为分数阈值，重复运行本格即可更新。人数上限不限制整段累计 track 数。
`MULTI_BBOXES=None` 自动检测；设为 `[[x1,y1,x2,y2], ...]` 则直接使用原图像素固定框，跳过检测器。每个框按输入顺序固定对应一个 track，重叠框也保留；数量由框列表决定，不受 MAX_PEOPLE 截断。逐视频 `bboxes=None` 可覆盖全局固定框切回自动检测。
每个框还可写成 `{'bbox': [x1,y1,x2,y2], 'start_seconds': 2.5}` 或 `{'bbox': [...], 'start_frame': 150}`，时间/源帧号二选一，起点均相对原视频，帧号从0开始。达到条件的第一个采样帧起才调用SAM；之前不导出该框的姿态帧，也不创建人物目录。普通坐标列表默认从0秒启用，可与字典混用。框的track ID按原列表顺序保留，不受启动顺序影响。
逐视频 `bboxes` 同样支持上述开始条件。若设定的开始点不在选定采样范围内，该框不会产生人物目录；总清单的 `not_started_track_ids` 会列出其ID。
手动框必须在原图范围内，人物需保持在对应区域；此模式不判断框内是否有人，不会自动跟随离开区域的人。批次所有路径和框范围在开始前统一检查。
默认 `MAX_FRAMES = None`、`STRIDE = 1`、`START_SECONDS = 0.0`、`END_SECONDS = None`，即处理完整视频的每一帧。`MAX_FRAMES = 30` 可先试跑每个视频的前 30 个采样帧；`STRIDE > 1` 会抽帧，不会补全跳过帧的姿态。

每次执行在 `OUTPUT_ROOT` 下创建独立的 `run_时间_随机后缀/`，每个视频再分配 `001_视频名/`、`002_视频名/`，同名视频或重复处理也不会覆盖。默认只保存 JSON：多人为总清单、诊断索引、相机摘要、逐帧耗时 `timings.jsonl` 及逐轨 `motion.json`，单人为 `motion.json`。仅 `SAVE_NPZ=True` 时额外创建多人 `track_0001/raw_npz/` 或单人 `motion_npz/`。`BATCH_RESULTS` 保存本次已完成视频的路径摘要。
整批处理完成后，本单元会将 `run_dir` 压缩成同名 ZIP，保存在 Notebook 当前工作目录，并显示可点击的下载链接。ZIP 保留批次目录及其内部相对路径；重复执行会为新批次生成独立压缩包。
路径列表会在推理前统一检查；运行途中报错则停止，之前完成的视频保留，失败视频可能留下未完成归档。修正后再次运行会写入全新的批次目录。

保留原视频 PTS 时间，不因抽帧或人物晚入场而压缩/归零。多人先按上一实际 bbox 的 IoU 一对一关联，再用预测框补充匹配；同帧高重叠框按分数去重。`TRACK_IOU_THRESHOLD` 是跨帧 IoU 阈值（达到即匹配），`DUPLICATE_IOU_THRESHOLD` 是同帧去重阈值（超过才抑制，1.0 关闭）。这不是身份重识别，交叉和遮挡可能切换 ID。每个视频的轨迹独立从 ID 1 开始；`MAX_MISSING_SECONDS` 应大于正常采样间隔。
默认不写原始 NPZ；开启后大数组逐帧归档。JSON 关键点仍占用随帧数增长的内存，长视频可用起止时间分段。各视频按顺序处理，不同时加载多个视频的姿态数据。

多人 `motion.json` 可逐人交给 `pose-blender`，也可将总 `manifest.json` 交给 `pose-blender-shared`。共享位置来自未标定的相机相对估计；overview 是展示总览，不是原镜头复原。导出成功请检查 `manifest.complete`，中断归档不能当作完整结果。只用 `allow_pickle=False` 读取可信 NPZ。

In [ ]:
# 修改此处后，只运行本单元即可继续处理视频，无需重新加载模型。
VIDEOS = PROJECT_DIR / "input.mp4"  # 单个文件，也支持字符串绝对路径
# 多个文件时，将上一行改为列表，例如：
# VIDEOS = [PROJECT_DIR / "video1.mp4", PROJECT_DIR / "video2.mp4"]
# 可按视频单独设置：
# VIDEOS = [{"video": PROJECT_DIR / "a.mp4", "max_people": 1},
#           {"video": PROJECT_DIR / "b.mp4", "bboxes": [[100, 100, 800, 1000], [900, 100, 1600, 1000]]}]
OUTPUT_ROOT = RUNTIME_DIR / "outputs"
MAX_PEOPLE = 20  # 自动检测人数上限；重复运行本格即可更新，无需重载模型
DETECTOR_THRESHOLD = 0.7  # 自动检测分数阈值，同样可在本格重新设置
MULTI_BBOXES = None  # 多人固定框，原图像素；None 自动检测，列表/字典写法见下面示例
# MULTI_BBOXES = [
#     {"bbox": [100, 100, 800, 1000], "start_seconds": 2.5},
#     {"bbox": [900, 100, 1600, 1000], "start_frame": 150},
#     [1700, 100, 2400, 1000],  # 未指定开始条件，默认0秒启用
# ]
STRIDE = 1
MAX_FRAMES = None  # 完整视频；试跑可改为 30（每个视频分别计数）
START_SECONDS = 0.0
END_SECONDS = None
TRACK_IOU_THRESHOLD = 0.25  # 跨帧实际 bbox 交并比达到此值则关联
DUPLICATE_IOU_THRESHOLD = 0.7  # 同帧重复框超过此交并比仅留高分框；1.0 关闭
MAX_MISSING_SECONDS = 1.0  # 应大于正常采样间隔 STRIDE / 源 FPS
SAVE_NPZ = False  # JSON 驱动不需要 NPZ；网格/原始参数分析时才开启
NPZ_COMPRESSED = True
INFERENCE_TYPE = "body"
CONTINUE_ON_POSE_ERROR = False
SINGLE_BBOX = None  # 单人模式: [x1, y1, x2, y2]；None 为全图

from datetime import datetime
import os
import tempfile
import numpy as np
from extract_video import extract
from multi_person import extract_multi, TorchvisionPersonDetector
from video_jobs import prepare_video_jobs

if "estimator" not in globals() or "runtime_metadata" not in globals():
    raise RuntimeError("请先执行前面的配置、准备和模型加载单元；重启 kernel 后也需要重新加载模型")
jobs = prepare_video_jobs(VIDEOS, max_people=MAX_PEOPLE,
    manual_bboxes=MULTI_BBOXES if MULTI_PERSON else None)
if not MULTI_PERSON and any(job["bboxes"] is not None for job in jobs):
    raise ValueError("bboxes 是多人模式设置；单人模式请使用 SINGLE_BBOX")
needs_detector = MULTI_PERSON and any(job["bboxes"] is None for job in jobs)
if needs_detector:
    if not 0 < DETECTOR_THRESHOLD <= 1:
        raise ValueError("DETECTOR_THRESHOLD 必须在 (0, 1] 范围")
    if globals().get("detector") is None:
        if not DETECTOR_WEIGHTS.is_file():
            raise FileNotFoundError("自动检测需要权重，请先运行检测器下载单元；手动框模式无需权重")
        first_auto = next(job for job in jobs if job["bboxes"] is None)
        detector = TorchvisionPersonDetector(DETECTOR_WEIGHTS, device=runtime_metadata["device"],
            score_threshold=DETECTOR_THRESHOLD, max_people=first_auto["max_people"])
    if detector.device != runtime_metadata["device"]:
        raise RuntimeError("SAM 与检测器必须使用同一设备，请重新执行模型加载单元以替换旧检测器")
videos = [job["video"] for job in jobs]

OUTPUT_ROOT = Path(OUTPUT_ROOT).expanduser().resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
run_dir = Path(tempfile.mkdtemp(prefix=datetime.now().strftime("run_%Y%m%d_%H%M%S_"), dir=OUTPUT_ROOT))
BATCH_RESULTS = []
print("本次输出目录:", run_dir)
with torch.inference_mode():
    for index, job in enumerate(jobs, start=1):
        video = job["video"]
        video_dir = run_dir / f"{index:03d}_{video.stem}"
        print(f"[{index}/{len(videos)}] {video} -> {video_dir}")
        if MULTI_PERSON:
            active_detector = None
            if job["bboxes"] is None:
                detector.configure(max_people=job["max_people"], score_threshold=DETECTOR_THRESHOLD)
                active_detector = detector
                print("自动检测:", detector.metadata)
            else:
                print("手动固定框:", job["bboxes"], "track ID 按列表顺序从 1 开始")
            result = extract_multi(video, video_dir, estimator, active_detector,
                manual_bboxes=job["bboxes"],
                stride=STRIDE, max_frames=MAX_FRAMES,
                start_seconds=START_SECONDS, end_seconds=END_SECONDS,
                iou_threshold=TRACK_IOU_THRESHOLD, max_missing_seconds=MAX_MISSING_SECONDS,
                duplicate_iou_threshold=DUPLICATE_IOU_THRESHOLD,
                inference_type=INFERENCE_TYPE, save_npz=SAVE_NPZ, npz_compressed=NPZ_COMPRESSED,
                continue_on_error=CONTINUE_ON_POSE_ERROR, metadata=runtime_metadata)
            if not result["complete"]:
                raise RuntimeError(f"未完成的多人结果: {video_dir}")
            json_files = [str(video_dir / person["json_file"]) for person in result["tracks"] if person["json_file"]]
            status, sampled_frames = result["status"], result["sampled_frames"]
            for person in result["tracks"]:
                print("track", person["track_id"], person["status"], person["json_file"])
        else:
            video_dir.mkdir()
            output_json = video_dir / "motion.json"
            result = extract(video, output_json, estimator, stride=STRIDE, max_frames=MAX_FRAMES,
                start_seconds=START_SECONDS, end_seconds=END_SECONDS, bbox=SINGLE_BBOX,
                inference_type=INFERENCE_TYPE, metadata=runtime_metadata,
                save_npz=SAVE_NPZ, npz_compressed=NPZ_COMPRESSED,
                continue_on_error=CONTINUE_ON_POSE_ERROR)
            json_files = [str(output_json)]
            status, sampled_frames = "ok", len(result["frames"])
        BATCH_RESULTS.append(dict(video=str(video), output_dir=str(video_dir),
                                  status=status, sampled_frames=sampled_frames, json_files=json_files))
        print("状态:", status, "采样帧:", sampled_frames, "JSON:", json_files)
        if SAVE_NPZ:
            candidates = (video_dir.glob("track_*/raw_npz/frame_*.npz") if MULTI_PERSON
                          else (video_dir / "motion_npz").glob("frame_*.npz"))
            first_npz = next(candidates, None)
            if first_npz is not None:
                with np.load(first_npz, allow_pickle=False) as raw:
                    print("NPZ 示例:", first_npz, {key: raw[key].shape for key in raw.files})
        del result  # 下一个视频开始前释放本视频的姿态数据，模型继续复用。
print("本次已完成视频数:", len(BATCH_RESULTS), "；输出目录:", run_dir)

# 将本次结果目录压缩为 ZIP，点击输出链接下载。
from pathlib import Path
import shutil
from IPython.display import FileLink, display

folder = Path(run_dir)
# 也可手动指定：folder = Path("/workspace/pose_pipeline/runtime/outputs/某个目录")
zip_path = shutil.make_archive(
    str(Path.cwd() / folder.name),
    "zip",
    root_dir=folder.parent,
    base_dir=folder.name,
)
print("ZIP:", zip_path)
display(FileLink(Path(zip_path).name))